# Embeddings and Vector Databases with Pinecone

**Embeddings** are vector representations of text. They are used to represent text in a vector space, where the distance between vectors represents the semantic similarity between the texts.

**Vector Databases** are databases that store vectors and their associated metadata. They are used to store and retrieve embeddings. Vector databases are organized into indexes (also called namespaces) - similar to tables in a relational database.

[Pinecone](https://www.pinecone.io/) is a vector database service that allows you to store and retrieve embeddings. It is a hosted service that allows you to scale your vector database as needed.

There are two main ways to use Pinecone:

1. **Store an embedding** - Store an embedding in a vector database.
    - Embed the text you want to store.
    - Create a document with the embedding and metadata.
    - Store the document in a vector database.
2. **Query a vector database** - Query a vector database for the most similar embeddings to a given query.
    - Embed the query.
    - Query the vector database with the embedded query.
    - Retrieve the most similar embeddings to the query.

### Install Pinecone and OpenAI

In [1]:
!pip install pinecone openai


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Import Libraries

In [2]:
import os
import uuid
from datetime import datetime, timezone
from pinecone import Pinecone
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

### Define Environment variables

In [3]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY") # Pinecone API key
PINECONE_INDEX_NAME=os.getenv("PINECONE_INDEX_NAME") # Name of the vector database index
PINECONE_NAMESPACE=os.getenv("PINECONE_NAMESPACE") # Namespace in your index on Pinecone.io
EMBEDDING_MODEL=os.getenv("EMBEDDING_MODEL") # OpenAI embedding model
EMBEDDING_DIMENSIONS=os.getenv("EMBEDDING_DIMENSIONS") # Number of dimensions in the embedding

### Initialize Pinecone and OpenAI

In [4]:
# Initialize Pinecone for vector database
pc = Pinecone(PINECONE_API_KEY)
# Initialize the vector database index
index = pc.Index(PINECONE_INDEX_NAME)
# Initialize OpenAI for embeddings 
client = OpenAI()

/Users/nmirabets/GitHub/memory-ai-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Store an embedding
-----------

In [24]:
string_to_store = 'I like the color blue.'

### Embed the string

In [25]:
# OpenAI embeddings
def get_embeddings(string_to_embed):
    response = client.embeddings.create(
        input=string_to_embed,
        model=EMBEDDING_MODEL,
        dimensions=int(EMBEDDING_DIMENSIONS)
    )
    return response.data[0].embedding

In [26]:
vector = get_embeddings(string_to_store)

In [27]:
print(f"Vector representation of {string_to_store}: \n", vector)

Vector representation of I like the color blue.: 
 [0.028344176709651947, -0.024379516020417213, -0.054422833025455475, 0.04052764177322388, 0.010591307654976845, -0.027387624606490135, 0.010270358994603157, -0.011956913396716118, -0.06675733625888824, -0.020314166322350502, 0.014662951231002808, 0.022592272609472275, -0.015455882996320724, 0.027312107384204865, 0.03486384078860283, 0.02937624789774418, -0.02670796774327755, -0.045864202082157135, 0.022617444396018982, -0.013353983871638775, -0.007878975942730904, -0.02751348540186882, -0.01682778261601925, 0.042642127722501755, 0.010194841772317886, -0.018275197595357895, 0.028293831273913383, 0.005921818315982819, -0.0023725032806396484, -0.02432917058467865, -7.940137584228069e-05, -0.01436088141053915, -0.0632835328578949, -0.05865180492401123, 0.02454313635826111, -0.04745006561279297, 0.0380607433617115, 0.039822813123464584, -0.008929925970733166, 0.002597481943666935, 0.005817982368171215, 0.013580535538494587, 0.06031318753957

### Define the vector metadata to store in the vector database

In [28]:
user_id = "1234"
current_time = datetime.now(tz=timezone.utc)

### Build the vector document to be stored

In [29]:
# Build document dictionary
documents = [
    {
        "id": str(uuid.uuid4()),
        "values": vector,
        "metadata": {
            "user_id": user_id,
            "timestamp": str(current_time),
            "payload": string_to_store,
        },
    }
]


### Store the vector document in the vector database

In [30]:
index.upsert(
    namespace=PINECONE_NAMESPACE,
    vectors=documents,
)

{'upserted_count': 1}

## 2. Query a vector database
-----------

In [36]:
query_string = "What is my favorite color?"
user_id = "1234"
top_k = 1 # This is the number of most similar embeddings to return

### Embed the query

In [45]:
vector = get_embeddings(query_string)

### Query the vector database for similar top_k embeddings + filters

In [46]:
response = index.query(
    vector=vector,
    filter={
        "user_id": {"$eq": user_id},
    },
    namespace=PINECONE_NAMESPACE,
    include_metadata=True,
    top_k=top_k,
)

In [47]:
response

{'matches': [{'id': '78db5356-332a-46d8-a97a-365f2f7d311c',
              'metadata': {'payload': 'I like the color blue.',
                           'timestamp': '2026-03-23 09:55:37.793823+00:00',
                           'user_id': '1234'},
              'score': 0.570209503,
              'values': []}],
 'namespace': 'https://data-memory-2xdin0a.svc.aped-4627-b74a.pinecone.io',
 'usage': {'read_units': 1}}

### Build the memories list

In [48]:
memories = []
if matches := response.get("matches"):
    memories = [m["metadata"]["payload"] for m in matches]
memories

['I like the color blue.']

In [4]:
from agent.tools import save_memory

memories = ['My favorite fruit is mango.', 'I like to spaguetti.']

save_memory(memories)

'2 memories saved successfully'

In [6]:
from agent.tools import load_memories

prompt = "What is my favorite food?"

load_memories(prompt)

['My favorite fruit is mango.', 'I like to spaguetti.']